# SPACECONS — Notebook 01: load MOSTA E9.5 and first base method

1. Load one MOSTA section (E9.5_E1S1)
2. Preprocess it once and save the processed file, so every later method uses the *same* input.
3. Run base method #1: **diffusion pseudotime on PCA** (aspatial — coordinates are never used).
4. Save the output in the canonical format: one pseudotime per cell, **rank-scaled to [0, 1]** (Step 1 of the pipeline).
5. Quick preview of the Spatial Redundancy Index: how much of this pseudotime can be predicted from position alone?

**Data setup:** download `E9.5_E1S1.MOSTA.h5ad` from https://db.cngb.org/stomics/mosta/ (Download page), then on Kaggle: *Add Input → Upload → New Dataset*.


In [1]:
import glob
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.stats import rankdata

sc.settings.verbosity = 1
np.random.seed(0)

# Finds the file wherever Kaggle mounted your uploaded dataset
path = glob.glob('/kaggle/input/**/E9.5_E1S1*.h5ad', recursive=True)[0]
print('Loading', path)
adata = sc.read_h5ad(path)
adata.var_names_make_unique()
print(adata)
print('obs columns :', adata.obs.columns.tolist())
print('layers      :', list(adata.layers.keys()))
print('obsm        :', list(adata.obsm.keys()))

ModuleNotFoundError: No module named 'pandas'

## 1. Look at the tissue
Check that `obsm['spatial']` exists and see which tissue annotations are available — we will use one of them as the **root** (where development starts).

In [ ]:
ANN = 'annotation' if 'annotation' in adata.obs.columns else None
if ANN:
    adata.obs[ANN] = adata.obs[ANN].astype('category')
    print(adata.obs[ANN].value_counts())
else:
    print('No annotation column found')

coords = np.asarray(adata.obsm['spatial'])[:, :2]
fig, ax = plt.subplots(figsize=(6, 7))
if ANN:
    for lab in adata.obs[ANN].cat.categories:
        m = (adata.obs[ANN] == lab).values
        ax.scatter(coords[m, 0], coords[m, 1], s=4, label=lab)
    ax.legend(fontsize=6, markerscale=3, bbox_to_anchor=(1, 1))
else:
    ax.scatter(coords[:, 0], coords[:, 1], s=4)
ax.set_aspect('equal'); ax.set_title('E9.5_E1S1 tissue'); plt.show()

## 2. Preprocess (done once, shared by all methods)
Use raw counts if they are stored in a layer, then the standard steps: normalise → log → highly variable genes → PCA → kNN graph.

In [ ]:
if 'count' in adata.layers:
    adata.X = adata.layers['count'].copy()
adata.layers['counts'] = adata.X.copy()

sc.pp.filter_genes(adata, min_cells=10)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.tl.pca(adata, n_comps=30, mask_var='highly_variable')
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
print(adata)

## 3. Base method #1: diffusion pseudotime on PCA (aspatial)
Pseudotime needs a **root**. Set `ROOT_TYPE` to a tissue label from the list above (leave `None` to use the most extreme cell on diffusion component 1).

Note: a whole E9.5 embryo contains many lineages at once, so one global pseudotime is crude. That is fine for building the pipeline; later we will restrict to one tissue region with a clearer progression.

In [ ]:
ROOT_TYPE = None   # e.g. 'Brain' — pick a label from the printed annotation list

sc.tl.diffmap(adata, n_comps=15)
dc1 = adata.obsm['X_diffmap'][:, 1]

if ROOT_TYPE is not None:
    idx = np.where(adata.obs[ANN] == ROOT_TYPE)[0]
    # root = the most extreme cell of that tissue along DC1
    root = idx[np.argmin(dc1[idx])] if np.median(dc1[idx]) < np.median(dc1) else idx[np.argmax(dc1[idx])]
else:
    root = int(np.argmin(dc1))
adata.uns['iroot'] = int(root)
sc.tl.dpt(adata)
print('Root cell:', adata.obs_names[root], '| tissue:', adata.obs[ANN].iloc[root] if ANN else '-')

# Canonical format (Step 1): rank-scale to [0, 1] so methods can be compared/aggregated
t = adata.obs['dpt_pseudotime'].values.astype(float)
t = np.where(np.isfinite(t), t, np.nanmax(t[np.isfinite(t)]))   # unreachable cells -> latest
adata.obs['t_dpt_pca'] = (rankdata(t) - 1) / (len(t) - 1)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 7))
sca = ax.scatter(coords[:, 0], coords[:, 1], c=adata.obs['t_dpt_pca'], s=4, cmap='viridis')
ax.scatter(coords[root, 0], coords[root, 1], c='red', s=80, marker='*', label='root')
plt.colorbar(sca, label='rank-scaled pseudotime'); ax.legend()
ax.set_aspect('equal'); ax.set_title('Base method 1: DPT on PCA (aspatial)'); plt.show()

## 4. Preview of the Spatial Redundancy Index (SRI)
SRI asks: *can this pseudotime be predicted from (x, y) position alone?* SRI ≈ 1 → it is just a spatial gradient; SRI ≈ 0 → position explains none of it.

This is a **rough preview** using cross-validated kNN regression on coordinates. The real SRI (Step 7) will use a thin-plate spline fitted to the *consensus* pseudotime and be tested against null models.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score

y = adata.obs['t_dpt_pca'].values
pred = cross_val_predict(KNeighborsRegressor(n_neighbors=30), coords, y,
                         cv=KFold(5, shuffle=True, random_state=0))
print(f"SRI preview (CV R^2 of pseudotime from position only): {r2_score(y, pred):.3f}")

## 5. Save outputs for the next notebooks
- `mosta_E95_processed.h5ad` — the shared preprocessed input for every base method.
- `pt_dpt_pca.csv` — this method's output in canonical form. Every later method will save a file with the same columns, which makes the consensus step simple.

(To reuse these in later notebooks: *Save Version*, then add this notebook's output as an input.)

In [ ]:
adata.write_h5ad('/kaggle/working/mosta_E95_processed.h5ad')
out = pd.DataFrame({'cell_id': adata.obs_names,
                    'x': coords[:, 0], 'y': coords[:, 1],
                    'pseudotime': adata.obs['t_dpt_pca'].values,
                    'method': 'dpt_pca'})
out.to_csv('/kaggle/working/pt_dpt_pca.csv', index=False)
out.head()